In [51]:
import requests
import json

from dataclasses import dataclass
from datetime import datetime

import sys
import os
from urllib.parse import urlsplit

# Get the current directory of the notebook
notebook_dir = os.getcwd()

# Get the parent directory (assuming the notebook is in folder B)
parent_dir = os.path.abspath(os.path.join(notebook_dir, os.pardir))

# Add the parent directory to sys.path
sys.path.append(parent_dir)

from monitoring import strategies as strats
from monitoring.strategies import URL, _get_content_with_css_selector

In [6]:
# load username and password from secrets
Kemono_login = ""
Kemono_password = ""

with open('secrets.json', 'r', encoding='utf-8') as secs:
    dict = json.load(secs)

    Kemono_login = dict["Kemono_login"]
    Kemono_password = dict["Kemono_password"]

    print(f'{hash(Kemono_login) = } \n{hash(Kemono_password) = } ')

hash(Kemono_login) = -180668472278056983 
hash(Kemono_password) = 2398354230024001518 


In [14]:
login_url = 'https://kemono.party/account/login'
fav_url = 'https://kemono.party/favorites'

data = {
    "username": f"{Kemono_login}",
    "password": f"{Kemono_password}",
}

with requests.session() as session:
    login_response = session.post(login_url, data=data)
    fav_response = session.get(fav_url)
    session.close()


In [21]:
print(f"{login_response.status_code = }")
print(f"{fav_response.status_code = }")
print("")

print(login_response.text[:256].strip())
print("")

print(fav_response.text[:256].strip())

login_response.status_code = 200
fav_response.status_code = 200

<!DOCTYPE html>
<html prefix="og: https://ogp.me/ns#">
  <head>
    <script>var page_data = {}</script>

      <script defer="" data-api="/api/event" data-domain="kemono.party" src="/static/bundle/js/global-bc726bbacac216680f49.bundle.js"></script>

<!DOCTYPE html>
<html prefix="og: https://ogp.me/ns#">
  <head>
    <script>var page_data = {}</script>

      <script defer="" data-api="/api/event" data-domain="kemono.party" src="/static/bundle/js/global-bc726bbacac216680f49.bundle.js"></script>


In [46]:
html = fav_response.text
cards = _get_content_with_css_selector(html, ".user-card")
print(f"{len(cards) = }")

print(fav_url)
from urllib.parse import urlparse as uparse
print(uparse(fav_url))

len(cards) = 50
https://kemono.party/favorites
ParseResult(scheme='https', netloc='kemono.party', path='/favorites', params='', query='', fragment='')


In [48]:
@dataclass
class CardInfo:
    name: str | None = None
    date_time: datetime | None = None
    service: str | None = None
    link: URL | None = None

    def to_json(self) -> str:
        json_dict = {
            'name': self.name,
            'date_time': None if self.date_time is None else self.date_time.isoformat(),
            'service': self.service,
            'link': self.link
        }
        return json.dumps(json_dict)

    @classmethod
    def from_json(cls, json_str: str) -> 'CardInfo':
        json_dict = json.loads(json_str)
        date_time = None if json_dict['date_time'] is None else datetime.fromisoformat(json_dict['date_time'])
        return cls(
            name=json_dict['name'],
            date_time=date_time,
            service=json_dict['service'],
            link=json_dict['link']
        )

    def __str__(self):
        return f"Kemono Profile Information:\n" \
            f"- Service: {self.service}\n" \
            f"- Name: {self.name}\n" \
            f"- Date and Time: {self.date_time}\n" \
            f"- Link: {self.link}"

base_url = "://".join(urlsplit(fav_url)[:2])

profiles = []
for card in cards:
    service = card.select_one('.user-card__service')
    if service is not None:
        service = service.text.strip()

    name = card.select_one('.user-card__name')
    if name is not None:
        name = name.text.strip()
    
    dt = card.select_one('time.timestamp')
    if dt is not None:
        dt = datetime.strptime(dt.text.strip(), "%Y-%m-%d %H:%M:%S.%f")

    link = card.attrs.get('href', None)
    if link is not None:
        link = base_url + '/' + link

    profile = CardInfo(name, dt, service, link)
    profiles.append(profile)
